# 1. Overview

Operator transaction log for model training and validation-output publication. The workflow trains on `train`, validates on `val`, writes validation generated-pose artifacts, records the held-out `test` split for `test_model.ipynb`, and publishes runtime outputs to Drive.

# 2. Operator Configuration

## 2.1 Repository and roots

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

## 2.2 Model workflow inputs

In [ ]:
MODEL_KEY = "base_direct"
MODEL_CONFIG_RELATIVE_PATH = Path("configs/modeling/base_direct.yaml")
MODEL_MANIFEST_FAMILY = "tiered:clean:included"
MODEL_RUN_MODE = "smoke"
MODEL_COMPUTE_PROFILE = "portable"
# For Colab A100 80GB runs:
# MODEL_COMPUTE_PROFILE = "colab_a100_80gb"
MODEL_AUXILIARY_OBJECTIVES = ()

## 2.3 Configuration review

In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Model key: {MODEL_KEY}")
print(f"Model config relative path: {MODEL_CONFIG_RELATIVE_PATH}")
print(f"Manifest family: {MODEL_MANIFEST_FAMILY}")
print(f"Run mode: {MODEL_RUN_MODE}")
print(f"Compute profile: {MODEL_COMPUTE_PROFILE}")
print(f"Auxiliary objectives: {MODEL_AUXILIARY_OBJECTIVES}")

# 3. Bootstrap Boundary

## 3.1 Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive project parent is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")


## 3.2 System packages

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")


## 3.3 Repository checkout

In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    !rm -rf "{PROJECT_ROOT}"
    _exit_code = _exit_code if "_exit_code" in globals() else 0
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("stale checkout removal failed")

!git clone "{REPO_URL}" "{PROJECT_ROOT}"
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("git clone failed")

%cd {PROJECT_ROOT}
!git checkout "{REPO_REF}"
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("git checkout failed")

!git rev-parse HEAD
_exit_code = _exit_code if "_exit_code" in globals() else 0
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("rev-parse HEAD failed")

## 3.4 Install dependencies

In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"

## 3.5 Add source tree to import path

In [ ]:
import sys

source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.insert(0, str(source_path))
print(f"Source path active: {source_path}")

## 3.6 Workflow API import

In [ ]:
from text_to_sign_production.workflows.foundation.review import display_review_sections
from text_to_sign_production.workflows.model import ModelWorkflow, ModelWorkflowConfig
from text_to_sign_production.workflows.model.progress import visible_model_progress_session
from text_to_sign_production.modeling.candidates.bootstrap import ensure_model_provider_registered

# 4. Runtime Console: Plan

## 4.1 Build workflow config

In [ ]:
model_config = ModelWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    model_key=MODEL_KEY,
    manifest_family=MODEL_MANIFEST_FAMILY,
    model_config_relpath=MODEL_CONFIG_RELATIVE_PATH,
    run_mode=MODEL_RUN_MODE,
    compute_profile=MODEL_COMPUTE_PROFILE,
    auxiliary_objectives=MODEL_AUXILIARY_OBJECTIVES,
)
print(f"Generated model run: {model_config.run_name}")

## 4.2 Instantiate workflow

In [ ]:
model_workflow = ModelWorkflow(model_config)
model_progress = visible_model_progress_session(None)
print("Model workflow instantiated.")

## 4.3 Plan runtime

In [ ]:
model_runtime_plan = model_workflow.plan_runtime()
print("Model runtime plan built.")

## 4.4 Review runtime plan

In [ ]:
display_review_sections(model_workflow.review_runtime_plan(model_runtime_plan))

## 4.5 Validate runtime plan

In [ ]:
model_workflow.validate_runtime_plan(model_runtime_plan)
print("Model runtime plan validated.")

## 4.6 Preflight readiness

Preflight checks configuration, Drive inputs, provider/objective compatibility, and expected output roots before restore or execution starts.

In [ ]:
model_preflight = model_workflow.run_preflight(model_runtime_plan)
display_review_sections(model_workflow.review_preflight(model_preflight))
if not model_preflight.ready:
    first_issue = model_preflight.blocking_issues[0] if model_preflight.blocking_issues else "unknown"
    raise RuntimeError(
        f"Model preflight failed with {len(model_preflight.blocking_issues)} blocking issue(s); "
        f"first issue: {first_issue}"
    )

## 4.7 Smoke execution protocol

Review the ordered smoke execution protocol before any restore, training, inference, or publish step is allowed to run.

In [ ]:
model_smoke_protocol = model_workflow.build_smoke_execution_protocol(model_preflight)
display_review_sections(model_workflow.review_smoke_execution_protocol(model_smoke_protocol))

# 5. Runtime Console: Restore and Verify

## 5.1 Execute runtime restore

In [ ]:
model_restore_result = model_workflow.restore_runtime(
    model_runtime_plan,
    progress_session=model_progress,
)
print("Model runtime restore complete.")

## 5.2 Review restore result

In [ ]:
display_review_sections(model_workflow.review_runtime_restore(model_restore_result))

## 5.3 Verify runtime

In [ ]:
runtime_verification = model_workflow.verify_runtime(model_runtime_plan)
print(f"Model runtime {runtime_verification.readiness_level.value} checked.")

## 5.4 Review runtime verification

In [ ]:
display_review_sections(model_workflow.review_runtime_verification(runtime_verification))

# 6. Runtime Console: Model Execution

## 6.1 Resolve research/provider

In [ ]:
if not model_restore_result.execution.succeeded:
    raise RuntimeError("Runtime restore failed; stopping before provider execution.")
research_resolution = model_workflow.resolve_research(model_runtime_plan.execution_inputs)
ensure_model_provider_registered(MODEL_KEY)
provider_resolution = model_workflow.resolve_provider(research_resolution.request)
print("Research and provider resolution complete.")

## 6.2 Review research/provider

In [ ]:
display_review_sections(model_workflow.review_research_resolution(research_resolution))
display_review_sections(model_workflow.review_provider_resolution(provider_resolution))

## 6.3 Load provider config

In [ ]:
if not provider_resolution.provider_available:
    raise RuntimeError("Provider is unavailable; stopping before stage planning.")
provider_config = model_workflow.load_provider_config(provider_resolution, research_resolution.request)
if provider_config.request != research_resolution.request:
    raise RuntimeError(
        "Provider config request does not match research request; "
        "request propagation is unsafe."
    )

## 6.4 Plan stages

In [ ]:
stage_plan = model_workflow.plan_model_stages(provider_config)
if stage_plan.stage_plan.request != provider_config.request:
    raise RuntimeError(
        "Stage plan request does not match provider config request; "
        "stage execution is unsafe."
    )
print("Model stage plan built.")

## 6.5 Review stage plan

In [ ]:
display_review_sections(model_workflow.review_provider_config(provider_config))
display_review_sections(model_workflow.review_provider_effective_config(provider_config))
display_review_sections(model_workflow.review_stage_plan(stage_plan))

## 6.6 Execute training + validation stages

In [ ]:
stage_execution = model_workflow.execute_model_stages(
    stage_plan,
    progress_session=model_progress,
)
if not stage_execution.execution.completed:
    raise RuntimeError("Model stage execution failed; stopping before artifact publication.")
print("Model training and validation stages complete.")

## 6.7 Review stage execution

In [ ]:
display_review_sections(model_workflow.review_stage_execution(stage_execution))

# 7. Runtime Console: Artifacts and Reports

## 7.1 Write model run metadata

In [ ]:
stage_artifact_receipts = model_workflow.materialize_stage_artifact_receipts(stage_execution)
objective_artifacts = model_workflow.write_model_objective_artifacts(stage_execution)
metadata_artifacts = model_workflow.write_model_run_metadata_artifacts(
    research_resolution,
    provider_config,
    stage_plan,
    stage_execution,
    objective_artifacts,
)
print(f"Model metadata written: {metadata_artifacts.run_metadata_path}")

## 7.2 Review metadata

In [ ]:
display_review_sections(model_workflow.review_stage_artifact_receipts(stage_artifact_receipts))
display_review_sections(model_workflow.review_model_objective_artifacts(objective_artifacts))
display_review_sections(model_workflow.review_model_run_metadata_artifacts(metadata_artifacts))

## 7.3 Write validation artifacts

In [ ]:
validation_artifacts = model_workflow.write_model_validation_artifacts(stage_execution)
print(f"Validation summary written: {validation_artifacts.summary_markdown_path}")

## 7.4 Review validation artifacts

In [ ]:
display_review_sections(model_workflow.review_model_validation_artifacts(validation_artifacts))

## 7.5 Write model reports

In [ ]:
report_artifacts = model_workflow.write_model_reports(stage_execution, validation_artifacts, objective_artifacts)
print("Model reports written.")
print(f"A100 readiness summary: {model_workflow.layout.reports.a100_readiness_summary_report_path}")

## 7.6 Review reports

In [ ]:
display_review_sections(model_workflow.review_model_reports(report_artifacts))

# 8. Runtime Console: Publish and Verify

## 8.1 Build publish plan

In [ ]:
publish_plan = model_workflow.build_publish_plan(
    metadata_artifacts=metadata_artifacts,
    validation_artifacts=validation_artifacts,
    report_artifacts=report_artifacts,
    objective_artifacts=objective_artifacts,
    stage_artifact_receipts=stage_artifact_receipts,
    stage_execution=stage_execution,
)
print("Model publish plan built.")

## 8.2 Review publish plan

In [ ]:
display_review_sections(model_workflow.review_publish_plan(publish_plan))

## 8.3 Execute publish

In [ ]:
publish_execution = model_workflow.execute_publish(
    publish_plan,
    progress_session=model_progress,
)
print("Model publish execution complete.")

## 8.4 Review publish execution

In [ ]:
display_review_sections(model_workflow.review_publish_execution(publish_execution))

## 8.5 Verify publish

In [ ]:
publish_verification = model_workflow.verify_publish(publish_execution)
print("Model publish verification complete.")

## 8.6 Review publish verification

In [ ]:
display_review_sections(model_workflow.review_publish_verification(publish_verification))

## 8.7 Build publish result

In [ ]:
publish_result = model_workflow.build_publish_result(publish_plan, publish_execution, publish_verification)
if not publish_result.verification.succeeded:
    raise RuntimeError("Publish verification failed.")

## 8.8 Review publish result

In [ ]:
display_review_sections(model_workflow.review_publish_result(publish_result))

# 9. Final Summary

In [ ]:
final_result = model_workflow.build_final_result(
    research=research_resolution,
    provider_available=provider_resolution.provider_available,
    runtime_verification=runtime_verification,
    stage_execution=stage_execution,
    metadata_artifacts=metadata_artifacts,
    validation_artifacts=validation_artifacts,
    objective_artifacts=objective_artifacts,
    report_artifacts=report_artifacts,
    publish_result=publish_result,
)
display_review_sections(model_workflow.review_final_operator_summary(final_result))
print(f"Published model run: {model_config.run_name}")
print(f"Model key: {model_config.model_key.value}")
print(f"Manifest family: {model_config.manifest_family.family_id}")
print(f"Run mode: {model_config.run_mode.value}")
print("Train split: train")
print("Validation split: val")
print("Test split: reserved for notebooks/test_model.ipynb")
print(f"Recommended CHECKPOINT_POLICY: {model_smoke_protocol.handoff.recommended_checkpoint_policy}")
print(f"Validation report path: {validation_artifacts.summary_markdown_path}")
print(f"Publish verification status: {publish_verification.succeeded}")
print("Paste into notebooks/test_model.ipynb:")
print(f'MODEL_RUN_NAME = "{model_smoke_protocol.handoff.paste_into_test_model["MODEL_RUN_NAME"]}"')
print(f'CHECKPOINT_POLICY = "{model_smoke_protocol.handoff.paste_into_test_model["CHECKPOINT_POLICY"]}"')
print('TARGET_SENTENCE_NAME = "<choose-source_sentence_name-from-test-split>"')